## Simple RAG

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")
print(f"Key loaded :{api_key[:8]}..." if api_key else "Key NOT found")


client = OpenAI()

Key loaded :sk-proj-...


In [4]:



MODEL = "gpt-4.1-nano"

response = client.chat.completions.create(
    model = MODEL,
    messages = [{
        "role" : "user",
        "content" : "In  one sentence, what is RAG in AI"
    }
    ],
)

print(response.choices[0].message.content)

RAG (Retrieval-Augmented Generation) in AI is a method that combines traditional language generation models with retrieval systems to enhance the accuracy and relevance of generated responses by incorporating external knowledge sources.


In [5]:
from pathlib import Path

KB_PATH = Path("..") / "knowledge-base"
print("Exists:", KB_PATH.exists())

for folder in sorted(KB_PATH.iterdir()):
    files = list(folder.glob("*.md"))
    print(f"{folder.name}: {len(files)} files")


Exists: True
company: 4 files
contracts: 32 files
employees: 32 files
products: 8 files


In [6]:
employee_files = sorted((KB_PATH / "employees").glob("*.md"))

print([f.stem for f in employee_files[:5]])

sample = employee_files[0].read_text(encoding="utf-8")
print(f"\nCharacters: {len(sample)}\n")
print(sample[:800])

['Alex Chen', 'Alex Harper', 'Alex Thomson', 'Amanda Foster', 'Avery Lancaster']

Characters: 2547

# HR Record

# Alex Chen

## Summary
- **Date of Birth:** March 15, 1990
- **Job Title:** Backend Software Engineer
- **Location:** San Francisco, California
- **Current Salary:** $115,000  

## Insurellm Career Progression
- **April 2020:** Joined Insurellm as a Junior Backend Developer. Focused on building APIs to enhance customer data security.
- **October 2021:** Promoted to Backend Software Engineer. Took on leadership for a key project developing a microservices architecture to support the company's growing platform.
- **March 2023:** Awarded the title of Senior Backend Software Engineer due to exemplary performance in scaling backend services, reducing downtime by 30% over six months.

## Annual Performance History
- **2020:**  
  - Completed onboarding successfully.  
  - Met expec


In [9]:
knowledge = {}

for file in (KB_PATH / "employees").glob("*.md"):
    name = file.stem
    parts = name.split()
    last_name = parts[-1].lower()
    text = file.read_text(encoding="utf-8")

    if last_name not in knowledge:
        knowledge[last_name] = []

    knowledge[last_name].append(text)

for file in (KB_PATH / "products").glob("*.md"):
    product_name = file.stem.lower()
    text = file.read_text(encoding="utf-8")

    if product_name not in knowledge:
        knowledge[product_name] = []

    knowledge[product_name].append(text)

print("Names stored:", len(knowledge))
print("Chen documents:", len(knowledge["chen"]))
print("Thompson documents:", len(knowledge["thompson"]))

Names stored: 38
Chen documents: 2
Thompson documents: 2


In [10]:
def get_relevant_context(question):
    question = question.lower()
    context = []

    for name in knowledge:
        if name in question:
            for text in knowledge[name]:
                context.append(text)

    return context

In [11]:
results = get_relevant_context("What is Alex Chen's job title?")
print("Documents found:", len(results))

results = get_relevant_context("Tell me about CarLLM")
print("Documents found:", len(results))

results = get_relevant_context("What is the weather today?")
print("Documents found:", len(results))

Documents found: 2
Documents found: 1
Documents found: 0


In [16]:
SYSTEM_PROMPT = """You are an assistant for the company Insurellm.
Answer questions using ONLY the context provided.
If the context contains several people or products that match the question,
briefly describe each one.
If the answer is not in the context at all, say you don't know."""


def answer_question(question):
    documents = get_relevant_context(question)

    context = ""
    for doc in documents:
        context = context + doc + "\n\n"

    user_message = "Context:\n" + context + "\nQuestion: " + question

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
    )

    return response.choices[0].message.content

In [17]:
print(answer_question("What is Alex Chen's job title?"))
print("---")
print(answer_question("What is the weather today?"))

Alex Chen's job title is Senior Backend Software Engineer.
---
I don't know.


In [18]:
print(answer_question("Which employee works in San Francisco?"))
print("---")
print(answer_question("Which Insurellm product is for car insurance?"))
print("---")
print(answer_question("Who is Chen?"))


I don't know.
---
I don't know.
---
There are two individuals named Chen:

1. **Alex Chen**: A Senior Backend Software Engineer at Insurellm, based in San Francisco, California. He has a strong career progression within Insurellm, participated in various projects, and is actively involved in professional development and company initiatives.

2. **Robert Chen**: A Senior Full Stack Engineer at Insurellm, also based in San Francisco, California. He has been with Insurellm since 2016, leads major platform releases, and is recognized for his technical leadership and mentorship.


In [15]:
docs = get_relevant_context("Who is Chen?")
print("Documents retrieved:", len(docs))

for doc in docs:
    print(doc[:60])
    print("...")

Documents retrieved: 2
# HR Record

# Alex Chen

## Summary
- **Date of Birth:** Ma
...
# HR Record

# Robert Chen

## Summary
- **Date of Birth:** 
...
